# **EDA**

In [1]:
import re
from pathlib import Path
from collections import Counter
import pandas as pd

In [2]:
df = pd.read_csv("data/raw/raw.csv", dtype={"id_artikel": str})
print(f"Artikel mentah: {len(df)}")
df.head(3)

Artikel mentah: 3979


,id_artikel,keyword,url,timestamp,judul,isi_mentah
0,777827,BCA,https://www.cnbcindonesia.com/market/202610201...,2026-10-20,"Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ...",Indeks Harga Saham Gabungan (IHSG) berhasil me...
1,777666,BCA,https://www.cnbcindonesia.com/market/202610192...,2026-10-19,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F...",Kepala Ekonom PT Bank Central Asia Tbk David E...
2,777293,BCA,https://www.cnbcindonesia.com/market/202610190...,2026-10-19,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara...",PT Bank Central Asia Tbk (BBCA) menginformasik...


### **Gabungkan keyword yang relevan menjadi 1**

In [3]:
# BCA dan BBCA jadi satu artike

POLA_BANK = {
    "BCA":     r"\bbca\b|\bbbca\b|bank central asia", 
    "BRI":     r"\bbri\b|\bbbri\b|bank rakyat indonesia|brilink",
    "BNI":     r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}

def deteksi_bank(teks):
    teks = (teks or "").lower()
    return ", ".join(b for b, pat in POLA_BANK.items() if re.search(pat, teks))

df["bank"] = (df["judul"].fillna("") + " " + df["isi_mentah"].fillna("")).map(deteksi_bank)
print("Distribusi jumlah bank yang disebut per artikel:")
print(df["bank"].map(lambda x: len(x.split(", ")) if x else 0).value_counts().sort_index())
print("Artikel tanpa bank :", (df["bank"] == "").sum())
df[["id_artikel", "keyword", "bank", "judul"]].head(5)

Distribusi jumlah bank yang disebut per artikel:
bank
0       4
1    1868
2     754
3     908
4     445
Name: count, dtype: int64
Artikel tanpa bank : 4


,id_artikel,keyword,bank,judul
0,777827,BCA,"BCA, BRI, Mandiri","Aksi Jual Asing Rp1,97 T di Sesi I, Saham BCA ..."
1,777666,BCA,BCA,"RI Alami Halusinasi Daya Beli, Ekonom Ungkap F..."
2,777293,BCA,BCA,"BCA Mau Buyback 19-21 Oktober, Karyawan Dilara..."
3,776824,BCA,"BCA, BRI, BNI, Mandiri","Asing Keluar Rp2,92 Triliun di Sesi I, Saham B..."
4,776759,BCA,"BCA, BRI, BNI, Mandiri",Ada Bank Yang Sudah Jual Dolar di Rp18.300


### **Pecah berita menjadi per kalimat**

beberapa singkatan seperti PT. dispesifik, agar tidak terpisah kalimatnya

In [6]:
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj",
             "dll", "dsb", "dkk", "a.l", "yakni"]   # <- bisa diubah: tambah singkatan bila perlu

def pecah_kalimat(teks):
    t = re.sub(r"\s+", " ", str(teks)).strip()
    for ab in SINGKATAN:
        t = re.sub(rf"\b{re.escape(ab)}\.", f"{ab}<DOT>", t)
    t = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", t)
    kalimat = re.split(r"(?<=[.!?])\s+", t)
    return [k.replace("<DOT>", ".").strip() for k in kalimat if k.strip()]

baris = []
for _, r in df.iterrows():
    for i, k in enumerate(pecah_kalimat(r["isi_mentah"])):
        baris.append({
            "id_artikel": r["id_artikel"],
            "id_kalimat": f"{r['id_artikel']}_{i}",
            "bank": r["bank"],
            "timestamp": r["timestamp"],
            "kalimat": k,
        })
kal = pd.DataFrame(baris)
kal["n_kata"] = kal["kalimat"].str.split().str.len()
print(f"Total kalimat mentah: {len(kal)} dari {len(df)} artikel")
print(f"Rata-rata kalimat/artikel: {len(kal)/len(df):.1f}")

Total kalimat mentah: 82737 dari 3979 artikel
Rata-rata kalimat/artikel: 20.8


### **Distribusi frekuensi kalimat**

In [7]:
print("Statistik jumlah kata per kalimat:")
print(kal["n_kata"].describe().round(1))

Statistik jumlah kata per kalimat:
count    82737.0
mean        19.2
std         13.8
min          1.0
25%         12.0
50%         17.0
75%         23.0
max        437.0
Name: n_kata, dtype: float64


In [8]:
print("Distribusi panjang kalimat (kata):")
bins = [0, 5, 10, 15, 20, 30, 50, 10000]
print(pd.cut(kal["n_kata"], bins=bins, right=False).value_counts().sort_index())

Distribusi panjang kalimat (kata):
n_kata
[0, 5)          1351
[5, 10)         8838
[10, 15)       21597
[15, 20)       20492
[20, 30)       21399
[30, 50)        7462
[50, 10000)     1598
Name: count, dtype: int64


### **Duplicate**

In [10]:
kal["kalimat"].duplicated().sum()

np.int64(17174)

In [11]:
kal[kal["kalimat"].duplicated() == True]

,id_artikel,id_kalimat,bank,timestamp,kalimat,n_kata
121,776155,776155_2,BCA,2026-10-14,"Sementara itu, untuk tenor menengah, seperti F...",19
122,776155,776155_3,BCA,2026-10-14,Sedangkan tenor panajang seperti FR0107 hanya ...,13
123,776155,776155_4,BCA,2026-10-14,"Dibanding tahun lalu, tepatnya hasil lelang pa...",27
124,776155,776155_5,BCA,2026-10-14,Sedangkan untuk tenor menengah seperti FR0109 ...,17
125,776155,776155_6,BCA,2026-10-14,Adapun untuk tenor panjang saat itu seperti FR...,23
...,...,...,...,...,...,...
82649,608753,608753_18,Mandiri,2025-02-07,Meskipun Kanada dan Meksiko mengalami penundaa...,10
82650,608753,608753_19,Mandiri,2025-02-07,Negeri Tirai Bambu kemudian membalas tarif imp...,8
82663,608753,608753_32,Mandiri,2025-02-07,CNBC INDONESIA RESEARCH,3
82670,608414,608414_6,Mandiri,2025-02-06,"Ditambah, dalam waktu dekat ini juga akan ada ...",19


In [12]:
print(kal.loc[kal["id_kalimat"] == "776155_2", "kalimat"].iloc[0])

Sementara itu, untuk tenor menengah, seperti FR0110 bid to cover ratio nya 1,94 kali dan FR0111 sebanyak 2,08 kali.


In [13]:
# Jumlah kalimat yang paling banyak terduplicate
dup_counts = kal["kalimat"].value_counts()
for s, c in dup_counts[dup_counts > 1].head(30).items():
    print(f"  ({c}x) {s[:80]}")

  (113x) 1.
  (99x) Selamat berinvestasi secara bijak.
  (76x) 2.
  (61x) 3.
  (54x) 4.
  (47x) Keputusan berinvestasi sepenuhnya berada di tangan masing-masing investor sesuai
  (44x) 5.
  (27x) Nasabah BRI Prioritas bisa mendapatkan BRI Prioritas Debit Card yang menyediakan
  (26x) 6.
  (23x) Mengutip Stockbit, berikut net foreign sell perdagangan Rabu!
  (18x) Kemudian, manfaat spesial bagi nasabah Mandiri Prioritas adalah 4x Livin'poin un
  (18x) "Dalam sekejap, seluruh mobil di garasi terbakar, termasuk juga seisi rumah.
  (18x) Keuntungan bagi nasabah prioritas Bank Mandiri dapat dirasakan sejak bergabung.
  (18x) "Perusahaan para cukong dan keluarga Soeharto merupakan sasaran utama pembakaran
  (18x) Yakni welcome bonus berupa cashback senilai Rp 1,5 juta setelah aktivasi dan tra
  (18x) Selain itu, ada perlindungan dari pembelian barang, perlindungan asuransi perjal
  (18x) Ltd.
  (18x) Mengutip Stockbit, berikut net foreign sell perdagangan Senin!
  (18x) Nasabah istimewa BNI 

beberapa kaliamt yang terduplicate merupakan kutipan dari seseorang / istilah umum, sehingga perlu didrop

### **Missing Text**

In [14]:
(kal["kalimat"].str.strip() == "").sum()

np.int64(0)

### **Kalimat dibawah 5**

In [15]:
n_pendek = (kal["n_kata"] < 5).sum()
print(f"Kalimat < 5 kata : {n_pendek} ({n_pendek/len(kal)*100:.1f}%)")

Kalimat < 5 kata : 1300 (1.6%)


In [18]:
pendek = kal[kal["n_kata"] <= 5]
print(f"Distribusi jumlah kata pada kalimat pendek: {dict(pendek['n_kata'].value_counts().sort_index())}")

Distribusi jumlah kata pada kalimat pendek: {1: np.int64(529), 2: np.int64(97), 3: np.int64(215), 4: np.int64(459), 5: np.int64(547)}


In [19]:
print("\n30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):")
for s in pendek["kalimat"].head(50).tolist():
    print("  |", s)


30 contoh kalimat < 5 kata (pastikan memang sampah, bukan relevan):
  | "Karena apa?
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | 2.
  | BCA menjadi yang terparah.
  | Prediksi itu kemudian benar terjadi.
  | Anthony tak berkutik.
  | (Kompas, 15 Mei 1998).
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | Pengalaman serupa dibagikan akun @mawarniseptiana.
  | laporannya ke official IG Dana.
  | Hati-hati akun palsu ya!" tulisnya.
  | 1.
  | 2.
  | Haryn dalam keterangannya, Jumat (25/9/2026).
  | 2.
  | 3.
  | Salah satunya, Mall Taman Anggrek.
  | 4.
  | 5.
  | 6.
  | 7.
  | 8.
  | 2.
  | Berikut kisah perjalanan keduanya.
  | "Yang pasti rekam jejak beliau.
  | Misbakhun membenarkan jadwal ini.
  | Warisan Jumbo Michael Hartono!
  | Belakangan, keyakinan itu seolah terbukti.
  | Belakangan, keyakinan itu seolah terbukti.
  | "Itu kan sovereign rating.
  | "Enggak.
  | Al

Ada beberapa kata dibawah 5, yang memang bermakna ("BCA menjadi yang terparah."). dan untuk kata - kata lain yang hanya 5 kata saja bermakna, sehingga filternya hanya akan menghapus data < 4

### **Frekuensi kata**

In [9]:
tokens = []
for k in kal["kalimat"]:
    tokens += re.findall(r"[a-zA-Z]+", str(k).lower())
freq = Counter(tokens)
print(f"Total token: {len(tokens)} | kata unik: {len(freq)}")
print("\n30 kata paling sering:")
for w, c in freq.most_common(30):
    print(f"  {w:18} {c}")

Total token: 1500251 | kata unik: 21107

30 kata paling sering:
  dan                29943
  rp                 29739
  yang               28640
  di                 23036
  saham              22109
  pt                 18538
  miliar             18036
  bank               17181
  tbk                16717
  pada               16578
  dengan             16358
  ini                15830
  dari               13549
  indonesia          13457
  dalam              11101
  menjadi            10168
  pasar              8892
  triliun            8730
  untuk              8638
  ihsg               8090
  itu                8065
  indeks             8029
  ke                 7796
  juga               7482
  sebesar            7407
  perdagangan        7188
  akan               6897
  tersebut           6320
  atau               6157
  hari               6133


### **Analisis stopword & casefolding**

In [21]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
stopwords = set(StopWordRemoverFactory().get_stop_words())

n_stop = sum(1 for t in tokens if t in stopwords)

contoh = kal["kalimat"].iloc[42]
print("asli        :", contoh[:90])
print("casefold    :", contoh.lower()[:90])

tanpa_stop = " ".join(w for w in re.findall(r"\w+", contoh.lower()) if w not in stopwords)
print("tanpa stop  :", tanpa_stop[:90])

asli        : Termasuk terkait dengan transaksi yang kini makin banyak beralih melalui ruang digital.
casefold    : termasuk terkait dengan transaksi yang kini makin banyak beralih melalui ruang digital.
tanpa stop  : termasuk terkait transaksi kini makin banyak beralih melalui ruang digital


## 7. Kesimpulan EDA


* **Jumlah data**: **3.979 artikel** → **82.895 kalimat mentah** (rata-rata **20,8 kalimat/artikel**). Hanya **4 artikel** tidak terdeteksi menyebut salah satu dari 4 bank.
* **Cangkupan bank per artikel**:    
    1 bank = 1.868 artikel, 2 bank = 754, 3 bank = 908, 4 bank = 445. Jadi **~53%** artikel menyebut lebih dari satu bank
* **Deskripsi kalimat**: 
    * median 17 kata, 
    * rata-rata 19.2 
    * (min 1, maks 437)
* **Kalimat < 5 kata**: **1.300 (1,6%)**. mayoritas kata di range tersebut adalah footer, penomoran list, dan pecahan kutipan.
* **Duplikat**: **17.373 kalimat identik (21,0%)**. Mayoritas terduplicate karena merupakan kutipan dari figure politik atau orang terknal sehingga beberapa kutipan sama, selain itu, ada beberapa garbage value seperti : 
    * kalimat disclaimer/sanggahan, promo nasabah prioritas (BRI/BNI/Mandiri Prioritas), 
    * penomoran list (1., 2....), 
    * kalimat 'Mengutip Stockbit...'. 
* **Missing/kosong**: **0**
* **Frekuensi kata**:    
    1,5 juta token, 21.107 kata unik. Didominasi stopword (dan, yang, di) + kata domain finansial (rp, saham, bank, miliar, ihsg, triliun).  **Tidak ada kata yang dibuang berdasarkan frekuensi.**
* **Stopword & casefolding**: ada beberapa kata berimbuhan. Casefolding tidak mempengaruhi, tapi perlu distandarisasi